# Phase 4 submission -- the 12-label pseudo-label model on the real test set

Runs the five Phase 4 fold checkpoints (`efficientnet_b0` + slice mean-pool, trained on Qwen3-4B
soft pseudo-labels over all 12 findings, pooled OOF 0.7779, gold transfer 0.7252) across the hidden
test set. Replaces the Phase 1 submission, which trained 4 labels and emitted a constant 0.5 for
the other 8 (LB 0.558).

**The whole risk here is preprocessing parity.** Phase 4 trained on the Phase 2 artifacts: 256px,
letterboxed to square, per-series normalized, priority-ordered, mirrored at load time by
`mirrors_in_plane`. `notebooks/phase1-submit/` reads raw DICOMs at 224px with none of that, so
reusing its loader would feed this model something it never saw. Instead each test study goes
through the *same two functions the training data went through*: `prep_study` (pure by design for
exactly this) with Phase 2's parameters, then `PreppedStudyDataset` with run 1's parameters. No
third copy of the pipeline, so nothing can drift.

Design decisions recorded before the run:
- **Prep at `max_series=4, k_slices=24, size=256`, read at `max_series=1, n_slices=16`.** Both
  halves must match their training-time counterpart, and prepping fewer series would not be
  equivalent: `select_series` falls back to arbitrary df order once the priority list is exhausted,
  so a `max_series=1` prep picks a different series than a `max_series=4` prep does for any study
  with no sagittal fluid-sensitive series.
- **Prep runs inside `predict_fn`.** `build_submission` wraps only that call in its try/except, so a
  study whose DICOMs fail to decode falls back to 0.5 instead of killing the submission. A run that
  errors on the hidden set is the classic way to lose this competition.
- **Mean of all 5 fold models.** Fold 1 scored 0.7264 on gold against the ensemble's 0.7252 -- at
  n=58 that is noise, and picking the best fold on 58 studies is selection bias. Inference cost is
  irrelevant for the accuracy slot.
- **Artifacts are deleted as they are consumed**, so peak disk is one study rather than
  ~1.4 MB x the whole test set, whose size we do not know in advance.
- Device is probed, not asserted: a submission that refuses to run without a GPU scores nothing.

What backs the prep path is not this notebook's 3 visible studies -- it is Phase 2 running
`prep_study` over all 4,407 training studies with zero failures.


In [ ]:
import glob, os, shutil, sys, tempfile, time

GIT_SHA = 'PENDING'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
CKPTS = sorted(glob.glob('/kaggle/input/**/knee_phase4_fold*.pt', recursive=True))
assert len(CKPTS) == 5, f'expected 5 fold checkpoints, found {len(CKPTS)}: {CKPTS}'

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

print('git sha      ', GIT_SHA)
print('src          ', SRC)
print('competition  ', COMP_DIR)
for p in CKPTS:
    print('checkpoint   ', p)

In [ ]:
import numpy as np
import pandas as pd
import torch

from knee.infer import LABEL_COLUMNS, build_submission
from knee.dataset import PreppedStudyDataset
from knee.model import KneeModel
from knee.prep import prep_study, save_study_npz

# Phase 2's prep parameters and run 1's read parameters -- see the header cell.
PREP_K, PREP_SIZE, PREP_MAX_SERIES = 24, 256, 4
MAX_SERIES, N_SLICES = 1, 16

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
TEST_ROOT = f'{COMP_DIR}/test_series'
print(f'{len(test_df)} test studies')

# Probe rather than assert: this is the scored submission, and a hard GPU
# requirement turns an odd accelerator assignment into a zero.
device = 'cpu'
if torch.cuda.is_available():
    try:
        _ = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as e:
        print(f'GPU present but unusable with this torch build: {e}')
print('device:', device)

# pretrained=False: internet is off, and every weight is about to be overwritten
# by the checkpoint anyway. weights_only=True -- our own state_dicts are plain
# tensors, so there is no reason to allow arbitrary unpickling.
models = []
for path in CKPTS:
    m = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS), pretrained=False)
    m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    models.append(m.to(device).eval())
print(f'{len(models)} fold models loaded')

In [ ]:
NPZ_DIR = tempfile.mkdtemp(prefix='knee_prep_')
prep_seconds, forward_seconds = [], []

def predict_one(study_uid):
    """prep -> npz -> the tested loader -> mean of 5 fold models. Anything that
    raises in here becomes a row of 0.5 via build_submission's fallback."""
    t0 = time.time()
    series_slices, meta = prep_study(
        study_uid, TEST_ROOT, test_series_df,
        k_slices=PREP_K, size=PREP_SIZE, max_series=PREP_MAX_SERIES)
    npz_path = os.path.join(NPZ_DIR, f'{study_uid}.npz')
    save_study_npz(npz_path, series_slices, meta)
    prep_seconds.append(time.time() - t0)

    try:
        dataset = PreppedStudyDataset([study_uid], NPZ_DIR,
                                      n_slices=N_SLICES, max_series=MAX_SERIES)
        image, _, _ = dataset[0]
        t1 = time.time()
        batch = image.unsqueeze(0).to(device)
        with torch.no_grad():
            probs = np.mean([torch.sigmoid(m(batch))[0].cpu().numpy() for m in models], axis=0)
        forward_seconds.append(time.time() - t1)
    finally:
        # keep peak disk at one study: the test set size is not known in advance
        os.remove(npz_path)
    return probs

wall_start = time.time()
submission = build_submission(test_df['StudyInstanceUID'].tolist(), predict_one)
wall_total = time.time() - wall_start

n_fallback = len(test_df) - len(prep_seconds)
print(f'total wall time for {len(test_df)} studies: {wall_total:.1f}s ({wall_total/60:.1f} min)')
if prep_seconds:
    print(f'prep    {np.mean(prep_seconds):.3f}s/study (p95 {np.percentile(prep_seconds, 95):.3f}s)')
if forward_seconds:
    print(f'forward {np.mean(forward_seconds):.3f}s/study for {len(models)} models')
print(f'studies that hit the 0.5 fallback: {n_fallback}')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission.head())

assert len(submission) == len(test_df), 'row count must match test studies exactly'
assert list(submission.columns) == ['StudyInstanceUID'] + LABEL_COLUMNS, 'column contract'
assert submission[LABEL_COLUMNS].isna().sum().sum() == 0, 'no NaNs allowed'
vals = submission[LABEL_COLUMNS].to_numpy()
assert (vals >= 0).all() and (vals <= 1).all(), 'all predictions must be in [0, 1]'
print('submission.csv passed shape/range checks')

# Cheap distribution sanity check: the model was trained against these positive
# rates, so predictions wildly off them mean something broke upstream of scoring
# (a preprocessing mismatch shows up here before it shows up on the LB).
TRAIN_POSITIVE_RATE = {
    'ACL': 0.0819, 'MCL': 0.0204, 'Medial Meniscus': 0.3186, 'Lateral Meniscus': 0.1037,
    'Medial OA': 0.0740, 'Lateral OA': 0.0245, 'PF OA': 0.0579, 'Effusion': 0.2616,
    'Synovitis': 0.1125, "Baker's": 0.1470, 'Contusion': 0.1028, 'Fracture': 0.0138,
}
print(f'\n{n_fallback} of {len(test_df)} studies fell back to 0.5 -- at a high fallback '
      'count the means below are dominated by that constant, not by the model')
print(f'{"label":20s} {"mean pred":>10s} {"train rate":>11s}')
for label in LABEL_COLUMNS:
    print(f'{label:20s} {submission[label].mean():10.4f} {TRAIN_POSITIVE_RATE[label]:11.4f}')